> **Note:** This environment variable is required for fully deterministic CuBLAS ops on CUDA >= 10.2 when `reproducible=True` is set below. Without it, PyTorch raises a `RuntimeError` instead of training deterministically. It must be set before `torch` is imported. See the [README FAQ](../../README.md#reproducibility-and-cublas_workspace_config) for details.

In [ ]:
%env CUBLAS_WORKSPACE_CONFIG=:16:8

# How To Use Imagix
Imagix is our implementation of a variational autoencoder for image data.  
This tutorial follows the structure of [`Getting Started - Vanillix`](./Vanillix.ipynb), but is much less extensive because our pipeline works similarly across different architectures. Here we focus only on Imagix specifics.

**AUTOENCODIX** supports far more functionality than shown here, so we’ll also point to advanced tutorials where relevant.  

**IMPORTANT**

> This tutorial only shows the specifics of the Imagix pipeline. If you're unfamiliar with general concepts,  
> we recommend following the [`Getting Started - Vanillix`](./Vanillix.ipynb) Tutorial first.

## What You'll Learn

You’ll learn how to:

1. **Initialize** the pipeline and run it. <br><br>
2. Understand the Imagix specific **pipeline steps**. <br><br>
3. Access the Imagix specific **results** (mus, sigma, kl/mmd losses). <br><br>
4. **Visualize** outputs effectively. <br><br>
5. Apply **custom parameters**. or Architecture <br><br>
6. **Save, load, and reuse** a trained pipeline. <br><br>


**Setting the Correct Path**

In [ ]:
import os

p = os.getcwd()
d = "autoencodix_package"
if d not in p:
    raise FileNotFoundError(f"'{d}' not found in path: {p}")
os.chdir(os.sep.join(p.split(os.sep)[: p.split(os.sep).index(d) + 1]))
print(f"Changed to: {os.getcwd()}")


## 1) Initialize the Imagix Pipeline
`Imagix` is a standard VAE implementation for image data. We don't allow different data modalities for `Imagix`.  
To run the pipeline we need to prepare two things:  

1. A directory with image files. We allow the following extensions: `".jpg", ".jpeg", ".png", ".tif", ".tiff"` (NOT case sensitive).<br><br>
2. An annotation file with metadata, where we map `sample_ids` to image paths. For this, we need to provide the name of the column where the image path information is stored. This is done via the `img_paths_col` config parameter.

### ❗❗Requirements ❗❗
The data for this tutorial is hosted on Hugging Face Hub ([autoencodix/mnist](https://huggingface.co/datasets/autoencodix/mnist)) and is downloaded (and extracted) automatically in the cell below on first run.
#### 1.1 The Dataset
Here we use a balanced subsample of the classic MNIST handwritten digits dataset: 2000 images, 200 per digit (0-9), 28x28 grayscale. The annotation column `label` gives the digit class for each image.  

**A Look Inside the Annotation File**:


In [ ]:
import os
import zipfile
import pandas as pd
import autoencodix as acx
from huggingface_hub import hf_hub_download
from autoencodix.configs.default_config import (
    DefaultConfig,
    DataConfig,
    DataCase,
    DataInfo,
)

IMGMAPPING = hf_hub_download(repo_id="autoencodix/mnist", repo_type="dataset", filename="mnist_mappings.txt")
images_zip = hf_hub_download(repo_id="autoencodix/mnist", repo_type="dataset", filename="mnist_images.zip")

IMGROOT = os.path.join("data/images/mnist_images/")
if not os.path.isdir(IMGROOT):
    with zipfile.ZipFile(images_zip) as zf:
        zf.extractall("data/images/")

anno_df = pd.read_csv(IMGMAPPING, sep="\t", index_col=0)
anno_df.head()

**Define Config and Run Pipeline**

In [ ]:
imgconfig = DefaultConfig(
    data_case=DataCase.IMG_TO_IMG,
    checkpoint_interval=25,
    epochs=250,
    reconstruction_loss="bce",
    beta=0.005,
    scaling="MINMAX",
    anneal_function="logistic-late",
    data_config=DataConfig(
        data_info={
            "IMG": DataInfo(
                file_path=IMGROOT,
                scaling="MINMAX",
                data_type="IMG",
            ),
            "ANNO": DataInfo(
                file_path=IMGMAPPING,
                data_type="ANNOTATION",
            ),
        },
    ),
)

imagix = acx.Imagix(config=imgconfig)
result = imagix.run()
backup_datset = result.datasets

## 2) Understand Imagix Specific Steps
Since `Imagix` is jus a `Varix` for images, there are no extra steps for this pipeline.

## 3) Access Imagix Specific Results
The `result` object follows our standard interface. Refer to [1] for more details.  
We don't have any `Imagix` specific results, but you can directly visualize the reconstructions as images, as shown in the code below:

[1] [Tutorials/DeepDives/PipelineOutputTutorial.ipynb](../DeepDives/PipelineOutputTutorial.ipynb)


In [ ]:
import matplotlib.pyplot as plt

sample_img = result.reconstructions.get(split="test", epoch=-1)
sample_img = sample_img[0, :, :, :]
sample_img = sample_img.squeeze()
sample_img.shape

plt.imshow(sample_img, cmap="grey")

## 4) Visualize Imagix Results
Since the results of `Imagix` are visually interpretable, we add an additional visualization:  
We show a grid of original images and reconstructed images with label information. See the code below for how to obtain this visualization.


In [ ]:
imagix.visualizer.show_image_recon_grid(result=imagix.result, n_samples=5)

**Standard Visualizations**  
We also offer the standard visualizations as for all other pipelines. You can pass one or more column names from the annotation data. These columns will be used to color the visualizations accordingly. In this example, we use the `label` parameter.


In [ ]:
imagix.show_result(params=["label"])

## 5) Customize Imagix
Via the `Config` object we can apply customize the pipeline by chaning the `loss function` or the number of `epoch` or preprocessing. To get an full overview of the adjustable parameters, please refere to [2].  
In Addition, there is a way to use another architecture for the image autoencoder. We currently offer two architectures:`ImageVAEArchitecture` and `ImageVAEFastArchitecture`.
The fast architecture does not has BatchNorm and inplace activation functions, which can speed up the training process. The `ImageVAEArchitecture` is the default architecture and based on  [Yang & Uhler](https://arxiv.org/abs/1902.03515).

We can simply pass the architecture as type to the `model_type` parameter of the `Imagix` pipeline.

[2] [Tutorials/DeepDives/ConfigTutorial.ipynb](../DeepDives/ConfigTutorial.ipynb).

In [ ]:
from autoencodix.modeling._imgfast_architecture import ImageVAEFastArchitecture
from autoencodix.modeling._imagevae_architecture import ImageVAEArchitecture


imgconfig = DefaultConfig(
    data_case=DataCase.IMG_TO_IMG,
    checkpoint_interval=25,
    epochs=250,
    reconstruction_loss="bce",
    beta=0.005,
    scaling="MINMAX",
    anneal_function="logistic-late",
    data_config=DataConfig(
        data_info={
            "IMG": DataInfo(
                file_path=IMGROOT,
                scaling="MINMAX",
                data_type="IMG",
            ),
            "ANNO": DataInfo(
                file_path=IMGMAPPING,
                data_type="ANNOTATION",
            ),
        },
    ),
)

imagix = acx.Imagix(config=imgconfig, model_type=ImageVAEFastArchitecture)
result = imagix.run()
backup_datset = result.datasets

## 6) Load Save and Reuse Imagix
There are not `Imagix` specific steps here. See the `Getting Started - Vanillix` for details. Below is a basic save/load usecase.

In [ ]:
import os
import glob

outpath = os.path.join("tutorial_res", "imagix.pkl")

imagix.save(file_path=outpath)

folder = os.path.dirname(outpath)
pkl_files = glob.glob(os.path.join(folder, "*imagix.pkl"))
model_files = glob.glob(os.path.join(folder, "*imagix.pth"))

print("PKL files:", pkl_files)
print("Model files:", model_files)

# the load functionality automatically will build the pipeline object out of the three saved files
imagix_loaded = acx.Imagix.load(outpath)

In [ ]:
result_predict = imagix_loaded.predict(data=backup_datset)

Visualize on the new predictions.

In [ ]:
imagix_loaded.show_result(params=["label"])

In [ ]:
imagix_loaded.visualizer.show_image_recon_grid(result=imagix_loaded.result, n_samples=5)